##### Important: markowitz_results_james/main.py should be run in Slurm to get results, the below is just for development purposes

In [1]:
import numpy as np
from scipy.optimize import minimize
import pickle
from datetime import datetime
from tqdm import tqdm
from mis_dro.portfolio import calculate_transaction_cost
from mis_dro.metrics import calculate_sharpe_ratio, calculate_sortino_ratio

In [1]:
using_ipynb = True

In [ ]:
# TODO: cite https://github.com/BorisForce/PyPortfolioModels/blob/main/Min_Mean_Variance/Min_Mean_Variance_model.py for code if necessary
def mean_variance_opt(Sigma: np.ndarray, mu: np.ndarray, risk_aversion: float):
    """
    Compute the mean-variance optimal portfolio weights subject to the constraints of no short-selling (weights >= 0)
    and full investment (sum(weights) = 1).
    """
    n = len(mu)
    
    def objective(w):
        return - (np.dot(w, mu) - 0.5 * risk_aversion * np.dot(w, Sigma @ w))
    
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    bounds = [(0, 1) for _ in range(n)]
    w0 = np.ones(n) / n
    
    res = minimize(objective, w0, method='SLSQP', bounds=bounds, constraints=constraints)
    return res.x

def do_markowitz(training_df, test_df, risk_aversion_hyperparameter, ratio_type, all_out_of_sample_costs_so_far, risk_free_rates, period_for_ratio_in_weeks, include_transaction_costs, prev_stock_figi_list, prev_portfolio_weighting, stock_figi_list):

    assert ratio_type in (None, "sharpe", "sortino")

    # Start a timer for measuring total time up to and including solve for window
    Sigma_mu_calculation_start = datetime.now()

    # For each stock, take the mean of the 13-weekly returns in the training set to get its expected 13-weekly return in the test period
    mu = training_df.mean(axis=0)

    # Create a covariance matrix (Sigma) out of the 13-weekly returns (make sure to do this properly, considering dimensionality etc.)
    Sigma = training_df.cov()

    # Start a timer for measuring solve time for window
    solve_start = datetime.now()

    # Supply Sigma, mu and a chosen risk-aversion (remember, you planned to trial values on the upper side of the 1-50 range) to get the portfolio weighting, x
    portfolio_weighting = mean_variance_opt(Sigma, mu, risk_aversion=risk_aversion_hyperparameter)

    # Stop the timer for measuring solve time for window to finalise it
    solve_time = (datetime.now() - solve_start).total_seconds()

    # Stop the timer for measuring total time for window to finalise it
    time_from_Sigma_mu_calculation_to_solving_inclusive = (datetime.now() - Sigma_mu_calculation_start).total_seconds()

    # Calculate the out-of-sample cumulative returns using the portfolio weighting and the test dataset--see how out_of_sample_cost is calculated in mis_dro/main.py
    out_of_sample_cost = test_df @ portfolio_weighting

    if include_transaction_costs:

        transaction_cost = calculate_transaction_cost(prev_stock_figi_list, prev_portfolio_weighting, stock_figi_list, list(portfolio_weighting))

        out_of_sample_cost.iloc[0] -= transaction_cost

    if ratio_type:

        all_out_of_sample_costs_so_far = all_out_of_sample_costs_so_far + list(out_of_sample_cost)

        ratio = {"sharpe": calculate_sharpe_ratio, "sortino": calculate_sortino_ratio}[ratio_type](all_out_of_sample_costs_so_far, risk_free_rates, period_for_ratio_in_weeks)

    results_object = {
        "portfolio_weighting": portfolio_weighting, # TODO: start saving in list format rather than as a numpy array, then modify james-portfolio.ipynb accordingly
        "solve_time": solve_time,
        "time_from_Sigma_mu_calculation_to_solving_inclusive": time_from_Sigma_mu_calculation_to_solving_inclusive,
        "out_of_sample_cost": out_of_sample_cost,
        "risk_aversion_hyperparameter": risk_aversion_hyperparameter,
        "all_out_of_sample_costs_so_far": all_out_of_sample_costs_so_far
    }

    if ratio_type:

        results_object[ratio_type] = ratio

    return results_object

def unpickle_data(filename):

    with open(filename, 'rb') as f:
        data = pickle.load(f)

    return data

def pickle_results(results, using_ipynb, save_filename):

    with open(f"/dcs/pg24/u5674159/mis-dro-code/markowitz_results_james/{'ipynb_' if using_ipynb else ''}{save_filename}.pkl", "wb") as f:
        pickle.dump(results, f)

def do_markowitz_run_without_validation(djia_windows_filename, lambdas, save, using_ipynb, save_filename):

    windows = unpickle_data(djia_windows_filename)

    results_for_all_lambas = {}

    for risk_aversion_hyperparameter in lambdas:

        results_for_each_window = []

        for window in tqdm(windows):
            training_df, test_df = window

            # NOTE: I have not included transaction costs below, in order to reflect (post refactor) how I got the existing non-validation data--I got the out of sample costs without including transaction costs, then included them in results processing in james-portfolio.ipynb
            results = do_markowitz(training_df, test_df, risk_aversion_hyperparameter, None, None, None, None, False, None, None, None)

            results_for_each_window.append(results)

        results_for_all_lambas[risk_aversion_hyperparameter] = results_for_each_window

    if save:

        pickle_results(results_for_all_lambas, using_ipynb, save_filename)

def get_single_holdout_dataset_sizes_same_ratio_as_train_test(training_df, test_df):

    single_holdout_num_training_observations = round(len(training_df) / (len(training_df) + len(test_df)) * len(training_df))

    single_holdout_num_validation_observations = len(training_df) - single_holdout_num_training_observations

    return single_holdout_num_training_observations, single_holdout_num_validation_observations

def get_stock_figi_list(training_df):

    # TODO: this whole thing with the use of get_window_of_train_and_test_dataframes is a bit of a fudge--the stock IDs should be saved along with the other results. Moreover, I don't know why I appended _{window index} to each of the FIGIs in the first place, so undo that and get rid of the splitting done below.

    return [col.split("_")[0] for col in training_df.columns]

def choose_risk_free_rates_for_validation(risk_free_rates: list[float], number_of_extra_data_at_start_of_risk_free_rates_reserved_for_validation: int, num_test_weeks: int, num_validation_dates_per_window: int, window_index: int):

    t = number_of_extra_data_at_start_of_risk_free_rates_reserved_for_validation
    v = num_validation_dates_per_window

    return risk_free_rates[t: t + num_test_weeks * window_index] + risk_free_rates[t + num_test_weeks * window_index - len(v): t + num_test_weeks * window_index]

def choose_risk_free_rates_for_testing(risk_free_rates: list[float], number_of_extra_data_at_start_of_risk_free_rates_reserved_for_validation: int, num_test_weeks: int, window_index: int):

    t = number_of_extra_data_at_start_of_risk_free_rates_reserved_for_validation

    return risk_free_rates[t: t + num_test_weeks * (window_index + 1)]

def do_markowitz_run_with_single_holdout_validation_using_ratio_as_metric(markowitz_djia_windows_filename, dro_djia_windows_filename, risk_free_rates_file, lambdas, ratio_type, save, using_ipynb, save_filename_excluding_ratio_type):

    markowitz_windows = unpickle_data(markowitz_djia_windows_filename)

    dro_windows = unpickle_data(dro_djia_windows_filename)

    results_for_each_window_with_its_best_lambda = []

    prev_stock_figi_list = []
    prev_portfolio_weighting = []

    risk_free_rates = unpickle_data(risk_free_rates_file)

    all_out_of_sample_costs_so_far = []

    for i, window in tqdm(enumerate(markowitz_windows)):

        markowitz_training_df, test_df = window

        stock_figi_list = get_stock_figi_list(markowitz_training_df)

        single_holdout_training_df_size, single_holdout_validation_df_size = get_single_holdout_dataset_sizes_same_ratio_as_train_test(markowitz_training_df, test_df)

        single_holdout_training_df = markowitz_training_df[: single_holdout_training_df_size]
        
        single_holdout_validation_df = dro_windows[i][0][- single_holdout_validation_df_size:]  # NOTE: doing this because of the need for weekly returns, not 13-week returns from markowitz_training_df, when it comes to validation data

        period_for_test_ratio_in_weeks = 156

        # TODO: note that 51 is specific to the risk free returns file in use (because it has 51 weekly risk-free rates up to and including the first rebalance date)
        risk_free_rates_for_validation = choose_risk_free_rates_for_validation(risk_free_rates, 51, 13, len(single_holdout_validation_df), i)
        risk_free_rates_for_testing = choose_risk_free_rates_for_testing(risk_free_rates, 51, 13, i)

        best_lambda, highest_validation_ratio = None, -float('inf')

        for risk_aversion_hyperparameter in lambdas:

            validation_ratio = do_markowitz(single_holdout_training_df, single_holdout_validation_df, risk_aversion_hyperparameter, ratio_type, all_out_of_sample_costs_so_far, risk_free_rates_for_validation, period_for_test_ratio_in_weeks - 13 + len(single_holdout_validation_df), True, prev_stock_figi_list, prev_portfolio_weighting, stock_figi_list)[ratio_type]

            if validation_ratio > highest_validation_ratio:
                best_lambda = risk_aversion_hyperparameter
                highest_validation_ratio = validation_ratio

        results_for_best_lambda = do_markowitz(markowitz_training_df, test_df, best_lambda, ratio_type, all_out_of_sample_costs_so_far, risk_free_rates_for_testing, period_for_test_ratio_in_weeks, True, prev_stock_figi_list, prev_portfolio_weighting, stock_figi_list)

        results_for_each_window_with_its_best_lambda.append(results_for_best_lambda)

        prev_stock_figi_list = stock_figi_list
        prev_portfolio_weighting = list(results_for_best_lambda["portfolio_weighting"])

        all_out_of_sample_costs_so_far = results_for_best_lambda["all_out_of_sample_costs_so_far"]

    if save:

        pickle_results(results_for_each_window_with_its_best_lambda, using_ipynb, f"{save_filename_excluding_ratio_type}_with_{ratio_type}")

In [ ]:
markowitz_djia_windows_filename = "/dcs/pg24/u5674159/mis-dro-code/james-data/windows_rebalance_dates_20080220_to_20250430_inclusive_every_13_weeks_markowitz.pkl"
dro_djia_windows_filename = "/dcs/pg24/u5674159/mis-dro-code/james-data/windows_rebalance_dates_20080220_to_20250430_inclusive_every_13_weeks.pkl"

lambdas = (
    0.5,
    5,
    50,
    500
)

# do_markowitz_run_without_validation(markowitz_djia_windows_filename, lambdas, False, using_ipynb, "results_for_multiple_lambdas")
do_markowitz_run_with_single_holdout_validation_using_ratio_as_metric(markowitz_djia_windows_filename, dro_djia_windows_filename, "/dcs/pg24/u5674159/mis-dro-code/james-data/risk_free_returns_weekly_2007-03-07_to_2025-07-30_inclusive.pkl", lambdas, "sharpe", False, using_ipynb, "results_for_best_lambdas_from_single_holdout_validation")
do_markowitz_run_with_single_holdout_validation_using_ratio_as_metric(markowitz_djia_windows_filename, dro_djia_windows_filename, "/dcs/pg24/u5674159/mis-dro-code/james-data/risk_free_returns_weekly_2007-03-07_to_2025-07-30_inclusive.pkl", lambdas, "sortino", False, using_ipynb, "results_for_best_lambdas_from_single_holdout_validation")

0it [00:00, ?it/s]

30 10
